In [99]:
from IPython.display import HTML

# Observable
obs = [
   "#4269D0FF", "#F0BD3CFF", "#FF5D45FF", "#6CC5B0FF", "#3CA951FF", "#FF8AB7FF",
   "#A463F2FF", "#97BBF5FF", "#9C6B4EFF", "#9498A0FF", "#1B1B1BFF"
]

html = "".join(
    f'<div style="display:inline-block;width:100px;height:30px;background:{c};margin:2px">{c}</div>'
    for c in obs
)

HTML(html)

In [117]:
preamble = """
The greatest songs of all time!!! (According to me)\\\n

::: {.callout-note title="Note"}
- One song per artist.
- Classical music is not eligible and will be covered in a separate list.
:::

"""

In [118]:
import pandas as pd
from urllib.parse import urlparse, parse_qs

INPUT_CSV = "data/songs.csv"
OUTPUT_QMD = "music/songs.qmd"

def extract_youtube_id(url):
    """
    Extract YouTube video ID from common URL formats.
    """
    parsed = urlparse(url)

    if "youtube.com" in parsed.netloc:
        return parse_qs(parsed.query).get("v", [None])[0]
    elif "youtu.be" in parsed.netloc:
        return parsed.path.lstrip("/")
    return None

df = pd.read_csv(INPUT_CSV)
df = df.sort_index(ascending=False)

print_lyric = False
print_mood_scores = True

with open(OUTPUT_QMD, "w", encoding="utf-8") as f:
    # Quarto header
    f.write("---\n")
    f.write("title: \"Song Rankings\"\n")
    f.write("format: html\n")
    f.write("toc: false\n")
    f.write("---\n\n")

    f.write(preamble)

    f.write("""<div class="form-check form-switch mb-3"><input class="form-check-input" type="checkbox" id="toggleSwitch" checked><label class="form-check-label" for="toggleSwitch">Show Mood Levels</label></div>""")

    max_mood_score = 5
    mood_colors = {"Sad": "bg-primary",
                   "Happy": "bg-success",
                   "Angry": "bg-danger",
                   "Dark": "bg-dark",
                   "Dreamy": "bg-warning"
                   }
    mood_colors = {"Sad": obs[0],
                   "Happy": obs[3],
                   "Angry": obs[2],
                   "Dark": obs[-1],
                   "Dreamy": obs[1],
                   "Passion": obs[6],
                   "Groovy": obs[8]
                   }

    for i in df.index:
        row = df.loc[i]
        ranking = i + 1
        artist = row["Artist"]
        title = row["Title"]
        album = row["Album"]
        year = row["Year"]
        url = row["URL"]
        genre = row["Genre"]
        lyric = row["Lyric"]

        video_id = extract_youtube_id(url)

        f.write(f"## {ranking}. {artist} - {title}\n")
        f.write(f"**Album:** *{album}* ({year})\\\n")
        f.write(f"**Genres:** {genre}\n\n")

        if pd.notna(lyric) and print_lyric:
            f.write(f"""<span style="color:#769FCD"><i>{lyric.replace(' / ', '\\\n')}</i></span><br><br>""")

        if print_mood_scores:
            score_string = "<div class='toggle-me'>"
            for col in df.columns:
                if col.startswith("Mood_"):
                    mood = col.split("_")[1]
                    score = int(df.loc[i, col])  # 0–5
                    if score > 0:
                        percent = score / 5 * 100
                        # progress bar won't display if not on one line
                        #score_string += f"""<div style="display:grid; grid-template-columns: 64px 160px; align-items:center; margin-bottom:0px;"><div>{mood}</div><div class="progress" style="height:14px;"><div class="progress-bar {mood_colors[mood]}" role="progressbar" style="width:{percent}%;"></div></div></div>"""
                        score_string += f"""<div style="display:grid; grid-template-columns: 70px 160px; align-items:center; margin-bottom:0px;"><div>{mood}</div><div class="progress" style="height:14px;"><div class="progress-bar" style="background-color: {mood_colors[mood]}; width:{percent}%;" role="progressbar"></div></div></div>"""
                        #score_string += "●" * score + "○" * (max_score - score)
            score_string += "<br></div>"
            f.write(score_string)

        # Video
        if video_id:
            video_string = "<details><summary>Listen here</summary>"
            video_string += f"""<div class="lite-youtube-style"><lite-youtube videoid="{video_id}"></lite-youtube></div></details>\n"""
            f.write(video_string)


    cols = [c for c in df.columns if c.startswith("Mood_")]
    moodmax = df[cols].sum().max()
    sumstring = ""
    for c in cols:
        mood = c.split("_")[1]
        percent = df[c].sum() / moodmax * 100
        sumstring += f"""<div style="display:grid; grid-template-columns: 70px 160px; align-items:center; margin-bottom:0px;"><div>{mood}</div><div class="progress" style="height:14px;"><div class="progress-bar" style="background-color: {mood_colors[mood]}; width:{percent}%;" role="progressbar"></div></div></div>"""
    f.write("## Overall Mood Distribution\n")
    f.write(sumstring)

print(f"Generated {OUTPUT_QMD}")


Generated music/songs.qmd


In [101]:
code = """<br><br>
```{python}
#| echo: false
import pandas as pd
import plotly.express as px
INPUT_CSV = "../data/songs.csv"
df = pd.read_csv(INPUT_CSV)
df = df.sort_index(ascending=False)

counts = (
    df.groupby(["Country", "CountryName"])
      .size()
      .reset_index(name="count")
).sort_values("count", ascending=False).rename(columns={"count": "Count"})

fig = px.bar(
    counts,
    x="Country",
    y="Count",
    hover_data=["CountryName"]
)

fig.update_layout(yaxis_title=None)
fig.show()
```
"""

# with open(OUTPUT_QMD, "a", encoding="utf-8") as f:
#     f.write(code)

In [102]:
import pandas as pd
import plotly.express as px
INPUT_CSV = "data/songs.csv"
df = pd.read_csv(INPUT_CSV)
df = df.sort_index(ascending=False)

counts = (
    df.groupby(["Country", "CountryName"])
      .size()
      .reset_index(name="count")
).sort_values("count", ascending=False).rename(columns={"count": "Count"})

fig = px.bar(
    counts,
    x="Country",
    y="Count",
    hover_data=["CountryName"]
)

fig.update_layout(yaxis_title=None)
fig.show()

In [103]:
fig = px.histogram(df, x="Year")
fig.update_layout(yaxis_title=None)
fig.show()